In [5]:
from pathlib import Path
import pandas as pd
import osmnx as ox

In [6]:
# Disable OSMnx caching for this geocoding run
ox.settings.use_cache = False

input_csv = Path("../output/clients.csv")
output_csv = Path("../output/client_coords.csv")

df = pd.read_csv(input_csv).copy()

def geocode_with_fallback(address: str):
    if pd.isna(address) or str(address).strip() == "":
        return pd.Series({"latitude": 0.0, "longitude": 0.0, "geocode_ok": False})

    query = str(address)
    if "netherlands" not in query.lower():
        query = f"{query}, Heerlen, Netherlands"

    try:
        lat, lon = ox.geocode(query)
        return pd.Series({"latitude": float(lat), "longitude": float(lon), "geocode_ok": True})
    except Exception:
        return pd.Series({"latitude": 0.0, "longitude": 0.0, "geocode_ok": False})

coords = df["address"].apply(geocode_with_fallback)
client_coords = pd.concat([df, coords], axis=1)

client_coords.to_csv(output_csv, index=False)

print(f"Saved: {output_csv.resolve()}")
print(f"Total rows: {len(client_coords)}")
print(f"Geocoded OK: {int(client_coords['geocode_ok'].sum())}")
print(f"Fallback 0/0 rows: {int((~client_coords['geocode_ok']).sum())}")
client_coords.head(5)

Saved: C:\Users\merli\source\reposSchool\CARAI-Artificial-Intelligence-group-2\output\client_coords.csv
Total rows: 100
Geocoded OK: 100
Fallback 0/0 rows: 0


,name,address,care_arrangement,preferences,time_window_start,time_window_end,care_hours,dogs,cats,smokes,latitude,longitude,geocode_ok
0,Client 1,"Schoolstraat 24, Heerlen",HBH Basic,afternoon,12:00,18:00,1.0,0,1,True,50.884828,5.975441,True
1,Client 2,"Laan van Hövell tot Westerflier 13, Heerlen",V&V,morning,08:00,12:00,1.5,0,0,False,50.883850,5.980078,True
2,Client 3,"Coriovallumstraat 7, Heerlen",Wash & Ironing,afternoon,12:00,18:00,2.0,2,0,False,50.885632,5.977367,True
3,Client 4,"Pattonstraat 42, Heerlen",HBH Basic,afternoon,12:00,18:00,1.0,0,0,False,50.872605,5.997245,True
4,Client 5,"Kanaalstraat 22, Heerlen",HBH Plus,morning,08:00,12:00,1.5,1,0,False,50.867337,6.003869,True


In [7]:
# Import the client_coords and display all with lat and long 0
client_coords = pd.read_csv(output_csv)
fallback_rows = client_coords[~client_coords["geocode_ok"]]
print(f"Total fallback rows: {len(fallback_rows)}")

# print all street names with lat long 0
print("Fallback addresses:")
for idx, row in fallback_rows.iterrows():
    print(f" - {row['address']}")
    

Total fallback rows: 0
Fallback addresses:
